# Récupérer le corpus de discours présidentiels

**Ce notebook doit tourner sur une machine avec accès internet** 

Schéma vérifié à partir d'un échantillon réel du fichier : les champs utiles sont `titre`, `url`, `domaine`, `prononciation` (date, format YYYY-MM-DD), `intervenants` (liste de `{nom, qualite, qualite_long}`), `type_emetteur`.

```bash
pip install requests beautifulsoup4 tqdm 
```

Source : [Métadonnées des Discours publics de Vie-publique.fr — data.gouv.fr](https://www.data.gouv.fr/datasets/metadonnees-des-discours-publics-de-vie-publique-fr)

In [ ]:
from datetime import datetime
import requests
from collections import Counter
from bs4 import BeautifulSoup
import time, re, os
from tqdm import tqdm
import glob

URL_METADONNEES = "https://www.data.gouv.fr/api/1/datasets/r/160fc156-2723-46ae-bbac-fd66f0b021e0"

## Configuration plage de dates

Modifie ces deux variables pour changer la période couverte.

In [ ]:
# Plage de dates a recuperer 
DATE_DEBUT = datetime(2011, 1, 1)     # ex: datetime(1974, 1, 1) pour remonter a Giscard d'Estaing
DATE_FIN = datetime.now()             # ex: datetime(2017, 5, 14) pour s'arreter a la fin d'un mandat

#Limit du nombre de discours, à adapter suivant la config de la machine (300 fichier pour un RTX 2070 avec 16 Go DE RAM)
NOMBRE_MAX_DISCOURS = 300 

print("Periode:", DATE_DEBUT.date(), "->", DATE_FIN.date())
print("Duree:", (DATE_FIN - DATE_DEBUT).days, "jours (~", round((DATE_FIN - DATE_DEBUT).days / 365, 1), "ans)")

Periode: 2011-01-01 -> 2026-07-27
Duree: 5686 jours (~ 15.6 ans)


In [ ]:
reponse = requests.get(URL_METADONNEES, timeout=60)
reponse.raise_for_status()
donnees = reponse.json()

liste_discours = donnees if isinstance(donnees, list) else donnees.get("data", donnees.get("discours", []))
print(len(liste_discours), "discours au total dans le fichier (toutes fonctions, toutes dates confondues)")

153428 discours au total dans le fichier (toutes fonctions, toutes dates confondues)


## Filtrage

`est_president()` vérifie deux choses : 
* Chaque intervenant du discours, en comparant sa `qualite` en égalité stricte (pas juste "contient") à "président de la république". exclut correctement les présidents étrangers rencontrés en entretien (ex. "Président de la République du Tchad", qui contiendrait la sous-chaîne mais n'est pas une égalité exacte)

* En filet de sécurité, les champs `domaine`/`type_emetteur` du document lui-même

In [3]:
def est_president(entree):
    for interv in (entree.get("intervenants") or []):
        qualite = (interv.get("qualite") or "").strip().lower()
        if qualite == "président de la république":
            return True
    domaine = (entree.get("domaine") or "").strip().lower()
    type_emetteur = (entree.get("type_emetteur") or "").strip().lower()
    return domaine == "président de la république" or type_emetteur == "président de la république"

def parser_date(entree):
    valeur = entree.get("prononciation")
    if not valeur:
        return None
    try:
        return datetime.strptime(valeur[:10], "%Y-%m-%d")
    except ValueError:
        return None

discours_filtres = []
for entree in liste_discours:
    if not est_president(entree):
        continue
    date_discours = parser_date(entree)
    if date_discours and DATE_DEBUT <= date_discours <= DATE_FIN:
        discours_filtres.append(entree)

print("=" * 60)
print(len(discours_filtres), "discours du President de la Republique entre", DATE_DEBUT.date(), "et", DATE_FIN.date())
print("=" * 60)
if discours_filtres:
    print("Exemple:", discours_filtres[0]["titre"][:100])

3378 discours du President de la Republique entre 2011-01-01 et 2026-07-27
Exemple: Conférence de presse de M. Emmanuel Macron, président de la République, sur les relations franco-all


## Répartition par année

In [ ]:
compte_par_annee = Counter(parser_date(e).year for e in discours_filtres if parser_date(e))
for annee in sorted(compte_par_annee):
    print(annee, ":", compte_par_annee[annee], "discours")

print("\nTotal:", len(discours_filtres), "discours sur", len(compte_par_annee), "annees")
if compte_par_annee:
    print("Moyenne:", round(len(discours_filtres) / len(compte_par_annee), 1), "discours/an")

2011 : 9 discours
2012 : 23 discours
2013 : 31 discours
2014 : 28 discours
2015 : 29 discours
2016 : 31 discours
2017 : 29 discours
2018 : 15 discours
2019 : 13 discours
2020 : 12 discours
2021 : 12 discours
2022 : 12 discours
2023 : 16 discours
2024 : 13 discours
2025 : 15 discours
2026 : 12 discours

Total: 300 discours sur 16 annees
Moyenne: 18.8 discours/an


## Faisabilité locale — repère avant de lancer le téléchargement complet

Ordre de grandeur pour l'extraction d'entités/relations (cours GraphRAG, 1 appel LLM par discours) avec Qwen2.5-1.5B :

| Nombre de discours | RTX 2060 (laptop, 6 Go) | RTX 2070 (PC fixe, 8 Go) |
|---|---|---|
| ~50 | confortable, quelques minutes | confortable, plus rapide |
| ~150 | correct, 15-30 min | correct, 10-20 min |
| ~300 | limite haute recommandée, 30-60 min | correct, 20-40 min |
| 1000+ | déconseillé en local | à éviter aussi, plutôt échantillonner ou Colab |

La RTX 2070 (desktop, pas de throttling thermique, VRAM et bande passante mémoire plus élevées) encaisse mieux les gros volumes et permet aussi d'utiliser un modèle un peu plus grand (ex. Qwen2.5-3B) pour une meilleure qualité d'extraction si tu veux tenter au-delà de 300.

Le RAG (chunking + embeddings) supporte des volumes bien plus grands sans souci sur les deux machines (le coût n'est pas par appel LLM), donc la vraie contrainte vient du cours GraphRAG.

In [11]:
if len(discours_filtres) > NOMBRE_MAX_DISCOURS:
    print(len(discours_filtres), "discours trouves, echantillonnage a", NOMBRE_MAX_DISCOURS,
          "(repartis dans le temps, pas seulement les plus recents)")
    pas = len(discours_filtres) // NOMBRE_MAX_DISCOURS
    discours_filtres = discours_filtres[::pas][:NOMBRE_MAX_DISCOURS]
else:
    print("Volume sous le seuil (", NOMBRE_MAX_DISCOURS, ") — pas d'echantillonnage necessaire")

print(len(discours_filtres), "discours retenus pour le telechargement du texte integral")

Volume sous le seuil ( 300 ) — pas d'echantillonnage necessaire
300 discours retenus pour le telechargement du texte integral


## Récupération du texte intégral depuis vie-publique.fr

In [ ]:
DOSSIER_SORTIE = "./discours-presidents/"
os.makedirs(DOSSIER_SORTIE, exist_ok=True)

def nettoyer_nom_fichier(titre, date):
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", titre.lower())[:60].strip("-")
    return date + "_" + slug + ".txt"

def extraire_texte_page(url):
    reponse = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0 (recherche pedagogique)"})
    reponse.raise_for_status()
    soup = BeautifulSoup(reponse.text, "html.parser")
    corps = soup.find("div", class_=re.compile("field--name-field-texte|article-content|content", re.I))
    if corps is None:
        corps = soup.find("article") or soup.body
    return corps.get_text(separator="\n", strip=True) if corps else ""

erreurs = []
for entree in tqdm(discours_filtres):
    url = entree.get("url")
    titre = str(entree.get("titre", "sans-titre")).replace("\r", " ").replace("\n", " ").strip()
    date_str = str(entree.get("prononciation", "0000-00-00"))[:10]
    nom_fichier = nettoyer_nom_fichier(titre, date_str)
    chemin = os.path.join(DOSSIER_SORTIE, nom_fichier)

    if os.path.exists(chemin):
        continue

    if not url:
        erreurs.append((titre, "pas d'URL"))
        continue

    try:
        texte = extraire_texte_page(url)
        if len(texte) < 200:
            erreurs.append((titre, "texte trop court (" + str(len(texte)) + " caracteres)"))
            continue
        with open(chemin, "w", encoding="utf-8") as f:
            f.write("Titre: " + titre + "\nDate: " + date_str + "\nSource: " + url + "\n\n" + texte)
    except Exception as e:
        erreurs.append((titre, str(e)))

    time.sleep(0.5)

print(len(os.listdir(DOSSIER_SORTIE)), "fichiers dans", DOSSIER_SORTIE)
print(len(erreurs), "erreurs")
for titre, erreur in erreurs[:10]:
    print("-", titre, "->", erreur)

100%|██████████| 300/300 [01:23<00:00,  3.60it/s]

300 fichiers dans ./discours-presidents/
296 erreurs
- Conférence de presse de M. Emmanuel Macron, président de la République, sur les relations franco-allemandes et les questions internationales, à Brühl, au château d'Augustusburg le 17 juillet 2026. -> texte trop court (44 caracteres)
- Déclaration de M. Emmanuel Macron, président de la République, sur le combat contre la peine de mort, Paris le 30 juin 2026. -> texte trop court (44 caracteres)
- Déclaration de M. Emmanuel Macron, président de la République, sur l'action internationale face à l'accroissement des déséquilibres économiques, à Paris le 11 juin 2026. -> texte trop court (44 caracteres)
- Déclaration de M. Emmanuel Macron, président de la République, sur la victoire du Paris Saint-Germain à la Ligue des Champions, Paris le 31 mai 2026. -> texte trop court (44 caracteres)
- Conférence de presse de M. Emmanuel Macron, président de la République, sur les relations  franco-africaines et la situation internationale, à Nairobi 

## Vérification finale

In [ ]:
fichiers = glob.glob(DOSSIER_SORTIE + "*.txt")
print(len(fichiers), "discours prets dans", DOSSIER_SORTIE)
if fichiers:
    with open(fichiers[0], encoding="utf-8") as f:
        print("\n--- Apercu du premier fichier ---")
        print(f.read()[:500])

300 discours prets dans ./discours-presidents/

--- Apercu du premier fichier ---
Titre: Déclaration de M. Nicolas Sarkozy, Président de la République, sur la révolution Internet, à Paris le 24 mai 2011.
Date: 2011-05-24
Source: https://www.vie-publique.fr/discours/182151-declaration-de-m-nicolas-sarkozy-president-de-la-republique-sur-la-re?egn-publisher=dila_vp&egn-name=informer_opendata_discours

Texte intégral
L'Histoire se souvient toujours de ces lieux vers lesquels, à un moment donné, toutes les forces créatives d'une époque semblent vouloir converger.
Aussi, c'est en f
